# AutoSocial AI — Z-Image-Turbo image generation

This is the upstream free-aistudio Kaggle workflow with an authenticated AutoSocial API bridge in its final cell.

You must sign in to Kaggle, upload this notebook, enable Internet and a GPU T4 accelerator, and run cells in order. Kaggle GPU sessions are temporary and subject to account quotas. Do not share this notebook output: the final cell prints a random API bearer token.

Model weights are downloaded at runtime and can be large. The image workflow is Z-Image-Turbo; video uses LTX-Video 2.3. Image and video use separate GPU sessions because the image and LTX video weights are not loaded together. Keep each session active only while using its feature.


### 🛠️ Step 1: Sync Repository & Setup Environment

In [ ]:
import os

# 1. Dynamically download or update the script repository
if not os.path.exists('/kaggle/working/free-aistudio'):
    print("📦 Initializing workspace: Cloning repository codebase...")
    !git clone https://github.com/airesearch-official/free-aistudio.git /kaggle/working/free-aistudio
else:
    print("🔄 Workspace active: Syncing latest repository updates...")
    !git -C /kaggle/working/free-aistudio pull

# 2. Install requirements
print("📥 Ensuring python requirements are met...")
!pip install -q -r /kaggle/working/free-aistudio/requirements.txt

# 3. Add to python paths to resolve library imports
import sys
sys.path.append('/kaggle/working/free-aistudio')

### 📥 Step 2: Download Model Weights & Engine Binary

Downloads Z-Image-Turbo GGUF model, Flux VAE, and Qwen3 text encoder.

In [ ]:
# Choose model preset
MODEL_PRESET = "Z-Image-Turbo-Q4"

from src.downloader import restore_binary, download_models

# 1. Download and restore the C++ compiled execution binary from GitHub Releases
restore_binary(repo="airesearch-official/free-aistudio", tag="v1.0.0")

# 2. Download model weights
download_models(preset=MODEL_PRESET)

### ⚡ Step 3: Start the C++ Backend API Server

This starts the `sd-server` in the background. It takes about 90 seconds to load the models into GPU VRAM.

In [ ]:
from src.server import start_server, tail_logs

# Launch the background inference engine
server_process = start_server(preset="Z-Image-Turbo-Q4")

# Print tail logs to confirm load status
print("\nRecent server engine logs:")
print(tail_logs())

### Step 4: Start the AutoSocial worker bridge

The final code cell starts an authenticated API bridge bound to loopback and a temporary HTTPS tunnel. It does not launch or publish the upstream Gradio UI. Copy the printed `OPEN_GPU_WORKER_URL` and `OPEN_GPU_WORKER_TOKEN` into your local AutoSocial `.env`; keep both secret/private, and keep this notebook running while you use this model mode.

The image and video notebooks run different model configurations. One URL/token pair configures the currently active worker; switch to the other notebook and replace the pair when changing modes.

In [ ]:
# AutoSocial AI secure generation bridge (instead of a public Gradio UI)
import os, re, sys, time, secrets, subprocess, threading, urllib.request
from pathlib import Path

# The AutoSocial app repository is public. This downloads only the bridge source;
# all generation remains on this Kaggle GPU session.
bridge_source = "https://raw.githubusercontent.com/majid801/auto-social-001/main/kaggle/autosocial_bridge.py"
bridge_target = "/kaggle/working/free-aistudio/src/autosocial_bridge.py"
urllib.request.urlretrieve(bridge_source, bridge_target)

# A fresh, high-entropy credential is generated each time the Kaggle session starts.
# Copy it to the AutoSocial server environment; never publish this notebook output.
bridge_token = secrets.token_urlsafe(48)
os.environ["AUTOSOCIAL_API_TOKEN"] = bridge_token
os.environ["AUTOSOCIAL_MODEL_KIND"] = "image"
from src.autosocial_bridge import serve
bridge_thread = threading.Thread(target=serve, kwargs={"host": "127.0.0.1", "port": 8765}, daemon=True)
bridge_thread.start()
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:8765/healthz", timeout=2).read()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Authenticated bridge did not start; inspect notebook output above.")

cloudflared = "/usr/local/bin/cloudflared"
if not Path(cloudflared).exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared,
    )
    os.chmod(cloudflared, 0o755)
tunnel_log = open("/kaggle/working/autosocial-cloudflared.log", "w")
tunnel = subprocess.Popen(
    [cloudflared, "tunnel", "--url", "http://127.0.0.1:8765"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
service_url = None
for _ in range(90):
    if tunnel.poll() is not None:
        raise RuntimeError("Cloudflare Tunnel exited. Check /kaggle/working/autosocial-cloudflared.log")
    try:
        log = Path("/kaggle/working/autosocial-cloudflared.log").read_text(errors="replace")
        matches = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
        if matches:
            service_url = matches[-1]
            break
    except OSError:
        pass
    time.sleep(1)
if not service_url:
    raise RuntimeError("No temporary HTTPS tunnel URL appeared; inspect autosocial-cloudflared.log")
print("\nAutoSocial GPU bridge is running. Copy both values into the AutoSocial server .env file:")
print(f"OPEN_GPU_WORKER_URL={service_url}")
print(f"OPEN_GPU_WORKER_TOKEN={bridge_token}")
print("Set MEDIA_GENERATION_PROVIDER=free-aistudio, then restart AutoSocial's Node server.")
print("The tunnel is temporary and protected by the random bearer token. Keep this Kaggle session active.")
